---
layout: post
permalink: /csa/unit_03/3_2
title: Impact of Program Design
showReadingTime: true
toc: true
---

## Impact of Program Design

Writing a program that runs is not the same as writing a program that works. This topic is about
what happens after the code compiles: whether it holds up under conditions you did not test, what
it does to the people who use it, and what rules apply when you build it out of someone else's
work.

### Learning Targets

- I can explain what system reliability means and why running once is not evidence of it.
- I can identify the edge cases that a method fails to handle, and add validation that rejects
  invalid input instead of returning a wrong answer.
- I can describe both beneficial and harmful effects of the same program.
- I can explain how a program built for one purpose can cause harm beyond its intended use.
- I can decide whether code I found may be reused, based on its license.

### Success Criteria

You have met the targets when you can:

1. Look at a method and name two inputs that would break it.
2. Write a guard clause that rejects invalid input before any assignment happens.
3. Explain why a silent wrong answer is worse than a crash.
4. Give a beneficial and a harmful effect of the same program, without treating them as a balance.
5. Decide, from a license, whether you may reuse a piece of code.

## Lesson Design: The LxD Cycle

This lesson was built with the Learning Experience Design cycle. The design work is documented here
because every choice below traces back to something observed about actual learners.

### Empathize

Watching classmates write their first validated methods, the same pattern showed up repeatedly:
they tested one input, it returned the expected value, and they moved on. When asked "what happens
if someone passes a negative number," the common answer was some version of *"why would they?"*

The misconception underneath it: **invalid input is something users do wrong, not something the
method is responsible for handling.** Testing is treated as confirmation that the code works, not
as an attempt to break it.

This is worth teaching because it produces no error. The code compiles, one test passes, and the
failure surfaces later in a context where nobody is looking for it.

### Define

**Point of View statement:**

> A CSA student who can already write a working method needs to see that a method is responsible
> for its own invalid input, because they currently test to confirm success rather than to find
> failure — which passes on the one input they tried and hides silent corruption.

**Learning goal:** Students will identify the edge cases a method fails to handle and write a guard
clause that rejects them, then connect that habit to the social and ethical consequences of
unreliable software.

### Ideate

**How Might We question:**

> How might we make a silent failure *visible* to someone whose code has never thrown an error, so
> that validation feels like part of writing the method rather than extra work afterward?

**Activity chosen:** a trace table with a deliberately wrong row. Students watch a method accept
-400 degrees and confidently report a status. Nothing crashes. The discomfort of the table's last
column does the teaching.

**Example chosen:** a `Thermostat` class. Small enough to hold in your head, and physical enough
that an impossible value is obviously impossible — no domain knowledge needed to know -400°F is
wrong. It also scales naturally into the social-impact and unintended-consequence sections, so one
example carries all four ideas of the topic.

### Prototype

The lesson draft, the worked example, the broken-then-fixed pair, and the practice tasks are in the
sections below.

### Test

Taught on the assigned teaching day. Feedback and the resulting revisions are documented in the
Revision Log near the end of this notebook.

```mermaid
flowchart LR
    Empathize --> Define --> Ideate --> Prototype --> Test --> Empathize
```

## Hook: The First Five Minutes

Before any code, put this on the board and nothing else:

> A thermostat app shipped last week. It has one method that sets the target temperature. It was
> tested, it passed, and it is running in forty thousand homes right now.
>
> Here is the method body: `targetTemp = temp;`
>
> What is the worst thing a user could type?

Take answers out loud for sixty seconds. Someone will say a huge number. Someone will say zero.
Push until someone says something that is not a number at all.

Then:

> Every single one of those is accepted. The app does not crash, does not warn, and reports a
> status as if nothing is wrong. It passed its test.

That is the lesson. Everything after it is how to not do that.

## The Running Example

Run the cell below to define `Thermostat`. Every example in this lesson modifies or extends it, so
keep it defined as you work through the notebook — if you restart the kernel, run it again.

A thermostat holds a target temperature and reports what the system should be doing. Simple enough
that you can hold all of it in your head, which is exactly what makes the failures easy to see.

In [ ]:
public class Thermostat {
    private String roomName;
    private double targetTemp;

    public Thermostat(String roomName, double targetTemp) {
        this.roomName = roomName;
        this.targetTemp = targetTemp;
    }

    public void setTargetTemp(double temp) {
        targetTemp = temp;
    }

    public double getTargetTemp() {
        return targetTemp;
    }

    public String getStatus() {
        if (targetTemp >= 78) return "COOLING";
        if (targetTemp <= 65) return "HEATING";
        return "IDLE";
    }
}

## Part 1: System Reliability

**System reliability** is a program performing its intended tasks as expected, under stated
conditions, without failure.

The phrase that matters is *under stated conditions*. A method that returns the right answer for
one input you happened to try has demonstrated almost nothing. Reliability is a claim about the
whole range of inputs the method might see.

### Worked Example

Predict all three outputs before you run this. Write your predictions down — the point of the
exercise is lost if you run first and rationalize afterward.

In [ ]:
Thermostat t = new Thermostat("Lab", 72);
System.out.println(t.getStatus());

t.setTargetTemp(80);
System.out.println(t.getStatus());

t.setTargetTemp(-400);          // below absolute zero
System.out.println(t.getStatus());

Output:

```
IDLE
COOLING
HEATING
```

### How Does This Work?

| Call | `targetTemp` after | `getStatus()` returns | Is this correct? |
|---|---|---|---|
| `new Thermostat("Lab", 72)` | 72.0 | `IDLE` | Yes — 72 is between the two thresholds |
| `setTargetTemp(80)` | 80.0 | `COOLING` | Yes — 80 is at or above 78 |
| `setTargetTemp(-400)` | -400.0 | `HEATING` | **No.** -400°F is below absolute zero |

The third row is the whole lesson. The program did not crash. It did not print a warning. It
accepted an impossible temperature and confidently reported a status for it. Every test passed,
and the object is now holding garbage.

This is why "it ran on my computer" is not a reliability claim. The failure is silent — and you
just produced it yourself, in a cell that ran without complaint.

### Popcorn Hacks

1. `getStatus()` uses `>= 78` and `<= 65`. Predict the return value for a target temp of exactly
   78, exactly 65, and exactly 71.5. Which of those three did you have to think hardest about, and
   why?
2. A classmate says the `-400` case proves the code has a bug in `getStatus()`. Explain why
   `getStatus()` is actually behaving correctly, and name the method that is really at fault.
3. List three input values you would test against `setTargetTemp` that are not in the example
   above. For each one, say what you expect to happen and why you chose it.

## Part 2: Broken Code, Then Fixed

Here is the same mutator, written the way most people write it the first time.

### The broken version

```java
public void setTargetTemp(double temp) {
    targetTemp = temp;
}
```

**What error does this produce?** None at compile time, and none at runtime. That is the problem.
This method has no way to say no. Passing `-400`, `9999`, or `Double.NaN` all succeed silently,
and the object is left holding a value that cannot be true.

A method that accepts impossible input and reports success is worse than one that crashes,
because a crash tells you where the problem is.

### The fixed version

Run this cell to replace the class with a guarded version. Everything else about it is identical —
only `setTargetTemp` changed.

In [ ]:
public class Thermostat {
    private String roomName;
    private double targetTemp;

    public Thermostat(String roomName, double targetTemp) {
        this.roomName = roomName;
        this.targetTemp = targetTemp;
    }

    public void setTargetTemp(double temp) {
        if (Double.isNaN(temp) || temp < 50 || temp > 90) {
            throw new IllegalArgumentException("Target temp out of range: " + temp);
        }
        targetTemp = temp;
    }

    public double getTargetTemp() {
        return targetTemp;
    }

    public String getStatus() {
        if (targetTemp >= 78) return "COOLING";
        if (targetTemp <= 65) return "HEATING";
        return "IDLE";
    }
}

**Why this works:** the method now states its own contract. Valid input is 50 through 90 inclusive,
and anything else is rejected at the moment it arrives rather than being discovered later by
whoever reads `getStatus()`.

Three details worth noticing:

- The check comes **first**. Validate before you assign, or you have already corrupted the object.
- `Double.isNaN(temp)` is a separate check because `NaN` fails every comparison. `NaN < 50` is
  `false` and `NaN > 90` is also `false`, so without that first clause `NaN` would slip straight
  through the range test.
- The message includes the offending value. "Out of range" tells you nothing at 2am; "out of range:
  -400.0" tells you where to look.

### Proving the guard runs first

The claim above — that validation happens *before* assignment — is easy to state and easy to
doubt. This cell proves it. Watch what `getTargetTemp()` reports after a rejected call.

In [ ]:
Thermostat t = new Thermostat("Lab", 72);

try {
    t.setTargetTemp(-400);
} catch (IllegalArgumentException e) {
    System.out.println("rejected -> " + e.getMessage());
}

System.out.println(t.getTargetTemp());   // did the object change?
System.out.println(t.getStatus());

The object still holds 72.0. The exception interrupted the method before the assignment line was
ever reached, so the rejection left the thermostat exactly as it was.

This is the difference between a method that *fails* and a method that *corrupts*. A corrupted
object keeps running with a bad value inside it, and the damage surfaces somewhere far away from
the line that caused it. A method that throws leaves the object in the last state you know was
valid.

### Experiment: why `isNaN` needs its own clause

The second bullet above says `NaN` defeats a range check. That is one of those claims that stays
abstract until you watch it happen. Run this.

In [ ]:
double bad = Double.NaN;

System.out.println("NaN < 50  -> " + (bad < 50));
System.out.println("NaN > 90  -> " + (bad > 90));
System.out.println("isNaN     -> " + Double.isNaN(bad));

// With only the range test, NaN slips through:
if (bad < 50 || bad > 90) {
    System.out.println("caught");
} else {
    System.out.println("NOT caught -- NaN would have been stored");
}

Both comparisons are `false`, so a guard built only from `<` and `>` concludes that `NaN` is inside
the valid range. It is not inside anything. `NaN` means "not a number," and comparing it to a
number is a meaningless question, so Java answers `false` to all of them — including `NaN != NaN`,
which is `true`.

The practical rule: **any range check on a `double` needs an explicit `isNaN` clause**, because the
range check alone cannot see it.

### Popcorn Hacks

1. Delete the `Double.isNaN(temp)` clause and predict what `setTargetTemp(Double.NaN)` does. Then
   explain, in one sentence, why `NaN` defeats a range check.
2. Rewrite the guard so the valid range is a pair of named constants instead of the literals 50 and
   90. What does that change make easier later?
3. The fixed version throws an exception. An alternative is to return a `boolean` saying whether
   the change succeeded. Give one argument for each approach.

## Part 3: Social, Economic, and Cultural Impact

Programs have effects on society, the economy, and culture, and **the same program can be both
beneficial and harmful at once.** This is not a balance where one cancels the other. Both are
simply true, and a designer is responsible for knowing both.

Take the thermostat out of the classroom and put it in fifty thousand homes as a connected device:

| Effect | Beneficial | Harmful |
|---|---|---|
| Energy use | Lower bills, less grid strain during peak hours | — |
| Automatic scheduling | Comfort without manual adjustment | Assumes a regular schedule; penalizes shift workers |
| Remote access | Adjust the house before you get home | Requires a smartphone and home internet |
| Usage data | Utilities can plan capacity | Occupancy patterns reveal when a house is empty |

Notice that no row is purely bad, and the harms are not bugs. Nothing in that right-hand column is
a defect to be fixed. They are consequences of design choices that were reasonable on their own
terms.

The questions that belong in the design process, not after launch:

- Who uses this?
- Who is excluded by the assumptions it makes?
- What behavior does it encourage?

### Popcorn Hacks

1. Pick one row from the table and propose a design change that reduces the harm. Then state what
   your change costs — every one of them costs something.
2. Add a fifth row for a program you actually use. Fill in both columns honestly.
3. The "requires a smartphone" row is an access issue, not a code issue. Explain why it still counts
   as an impact of *program design*.

## Part 4: Unintended Consequences

A program written to solve a real problem can produce harmful effects **beyond its intended use.**
Good intentions are not a defense, because the harm usually does not come from the goal. It comes
from scale, or from a context nobody imagined.

Suppose the connected thermostat adds one sensible feature: during a heat wave, it nudges every
target temperature up by two degrees to reduce grid load.

- For one house, this is a barely noticeable adjustment.
- For fifty thousand houses, the demand drop is large, sudden, and simultaneous — a shock the grid
  was not designed for.
- For a household with a medically vulnerable resident, two degrees is not a nudge.

Nobody wrote a feature to endanger anyone. The feature does exactly what it was designed to do.
The harm comes from the same behavior repeated at a scale nobody tested.

**What follows from this:** shipping is not where your responsibility ends. Programs need ongoing
evaluation of their real effects, and a designer should be willing to change or remove a feature
that turns out to cause harm — including one that works perfectly.

### Popcorn Hacks

1. Explain the difference between a *bug* and an *unintended consequence*, using the two-degree
   feature as your example.
2. Propose one safeguard for the heat-wave feature. State what it would have to know about the
   household in order to work, and whether collecting that is itself a harm.
3. Name a program you use whose behavior would change meaningfully if it had a hundred times as
   many users.

## Part 5: Intellectual Property and Code Reuse

Programmers reuse other people's code constantly, and that is normal and good. The rules are about
*which* code and *under what terms.*

| What you found | May you use it? |
|---|---|
| Published as open source, free to use | Yes — and follow its license terms |
| Not open source | Only with permission, sometimes with payment |
| No license stated at all | Treat it as **not** free to use |

### The trap

**Being able to read code does not mean you may use it.** A public repository, a forum answer, a
snippet in a blog post — all visible, none automatically yours. Visibility is about access.
Licensing is about permission. They are unrelated.

When there is no license, the safe move is to find an alternative or write your own. "It was on
the internet" has never been a defense.

### Applied to the running example

Suppose you find a `TemperatureValidator` class in a public repo that handles the `NaN` and range
checks more thoroughly than yours. Before pasting it into `Thermostat`:

- Does the repo have a LICENSE file? If not, you may not use it.
- If it does, does that license let you use the code in the way you intend, and does it require
  you to credit the author or publish your own source?
- If the code is not open source, you need permission from the owner first.

### Popcorn Hacks

1. A classmate argues that because a snippet is only four lines long, licensing does not apply.
   Respond.
2. Explain the difference between *open source* and *publicly visible* to someone who thinks they
   are the same thing. Use one example of each.
3. You find exactly the method you need, with no license. List your options in order of preference
   and justify the ordering.

## Optional Extension

Beyond the scope of the exam, but worth knowing:

**Not all open source licenses grant the same permissions.** Permissive licenses generally let you
use the code in a closed-source project. Copyleft licenses may require that anything you build with
it also be published under the same terms. Both are open source; they obligate you differently.

**Validation is not the only reliability tool.** Automated tests check behavior across many inputs
every time the code changes, so a fix stays fixed. Assertions document assumptions inside the code
itself. Logging records what actually happened in production, where you cannot attach a debugger.

**Exceptions are part of a method's contract.** A method that throws on invalid input is making a
promise about what it will not silently accept. Documenting that promise is as much a part of the
method as the code inside it.

## Practice: Trace and Debug

### Predict the output

Given the **fixed** version of `setTargetTemp` (the one that throws), what does this print?

```java
Thermostat t = new Thermostat("Lab", 72);
try {
    t.setTargetTemp(95);
} catch (IllegalArgumentException e) {
    System.out.println("rejected");
}
System.out.println(t.getTargetTemp());
System.out.println(t.getStatus());
```

**A.** `rejected`, then `95.0`, then `COOLING`

**B.** `rejected`, then `72.0`, then `IDLE`

**C.** `95.0`, then `COOLING`

**D.** `rejected`, then `72.0`, then `COOLING`

Commit to an answer before checking. You have already run something very close to this, so this is
recall, not guesswork.

### Apply the Idea

Finish `setRoomName` in the cell below so that it is reliable by the standard used in this lesson.
The tests underneath it are already written: the first attempt should be accepted and the next
three should be rejected. Right now all four are accepted, which is the bug you are fixing.

In [ ]:
public class Room {
    private String roomName;

    public Room(String roomName) { this.roomName = roomName; }

    public void setRoomName(String name) {
        // TODO: reject null, empty, and whitespace-only names.
        // Validate BEFORE assigning, the same way setTargetTemp does.
        roomName = name;
    }

    public String getRoomName() { return roomName; }
}

// --- tests: the first should be accepted, the next three rejected ---
Room r = new Room("Lab");
String[] attempts = { "Nursery", null, "", "   " };

for (String a : attempts) {
    try {
        r.setRoomName(a);
        System.out.println("accepted -> [" + r.getRoomName() + "]");
    } catch (IllegalArgumentException e) {
        System.out.println("rejected -> " + e.getMessage());
    }
}

Then write a short answer for each:

1. What inputs must it reject, and why? Consider `null` and the empty string specifically.
2. Give one beneficial and one harmful effect of letting users name their own rooms in a connected
   thermostat product used by thousands of households.
3. You want to use a `StringUtils.isBlank()` helper you found in a public GitHub repo with no
   LICENSE file. State what you are allowed to do and what you will do instead.

## Answer Check

### Predict the output — answer: B

`95` is outside the valid range of 50 to 90, so the guard throws before the assignment line is
reached. That is the key point: **the assignment never happens.** The `catch` block prints
`rejected`, and the object still holds the value it was constructed with, 72.0. Since 72 is between
the thresholds, `getStatus()` returns `IDLE`.

- **A** assumes the value was stored before the exception was thrown. It was not — the guard runs
  first, which is exactly why order matters.
- **C** assumes no exception was thrown at all, which would be the behavior of the *broken* version.
- **D** correctly keeps 72.0 but then reports `COOLING`, which contradicts it. At 72.0,
  `getStatus()` returns `IDLE`.

### Apply the Idea — discussion

**1.** Reject `null`, because any later call to a `String` method on it throws
`NullPointerException` far away from the real cause. Reject the empty string and whitespace-only
strings, because a room with no name cannot be identified in a list. A length cap is also
reasonable. The general principle: a mutator should reject anything that leaves the object in a
state it cannot function in.

**2.** Beneficial: names make a multi-room system usable, since "Nursery" is meaningful where
"Device 4" is not. Harmful: user-supplied names travel — into logs, support tickets, and shared
dashboards — and people put personal information in them. A name like "Grandma's room" is data the
designer never asked for but is now storing.

**3.** With no LICENSE file, you have no permission to copy it, regardless of how small or useful it
is. Options in order: write the check yourself, since it is a few lines; use a library whose license
clearly permits your use; or contact the author for permission. Visibility is not permission.

## Quick Review

- **System reliability** means performing as expected under stated conditions without failure — not
  "it ran once."
- Test with a **variety of conditions**, especially edges: zero, negative, empty, `null`, `NaN`, and
  values exactly at a boundary.
- A method that **silently accepts invalid input** is more dangerous than one that crashes.
- Validate **before** assigning, or the object is already corrupt.
- Programs affect society, the economy, and culture, and the **same program can be both beneficial
  and harmful.**
- **Unintended consequences** are harms beyond intended use. They usually come from scale or an
  unimagined context, not from bad intentions.
- **Open source published as free to use** may be reused under its terms; **non-open-source** code
  needs permission and sometimes payment; **no license** means no.
- **Publicly visible is not free to use.**
- On the exam, name the specific concept and tie it to the scenario. "Technology affects society"
  earns nothing.

## Homework Hack

**Task.** Take a class you have already written for this course — any class with at least one
mutator — and make it reliable.

1. List every parameter in the class that is currently accepted without checking.
2. For each one, write down the inputs that should be rejected and why.
3. Add guard clauses. Validate before assigning.
4. Write a short `main` that attempts one valid and one invalid value for each mutator, and prove
   the object is unchanged after a rejection.

**Then answer, in 3–4 sentences each:**

- Pick one of your guards. What would go wrong, specifically, if this class shipped without it and
  ten thousand people used it? Name a concrete consequence, not "it would break."
- Did you reuse any code to write your validation — a snippet, a helper, anything found online? If
  so, state where it came from and what its license allows. If not, say how you would have checked.

**Submit:** your `.java` file, the output of your `main`, and the written answers.

## Grading Plans

### Popcorn Hacks — 1 point

Applies to the assigned popcorn set. Each set has three prompts.

| Criterion | Points |
|---|---|
| Prediction is stated *before* reasoning, and is specific (an actual value or outcome, not "it breaks") | 0.3 |
| Explanation identifies the mechanism, not just the result | 0.4 |
| Third prompt extends the idea to a case not given in the lesson | 0.3 |
| **Total** | **1.0** |

Partial credit is the intent. A student who predicts correctly but explains vaguely has understood
the behavior and missed the reasoning — that is 0.5, not a zero. A student who predicts wrong but
explains their reasoning clearly earns the explanation points, because a wrong prediction with
visible reasoning is more useful to me than a right one without it.

### Homework Hack — 1 point

| Criterion | Points |
|---|---|
| Every unvalidated parameter in the class is identified | 0.2 |
| Guard clauses are placed before assignment and reject the right inputs | 0.25 |
| `main` demonstrates both acceptance and rejection, and proves the object is unchanged after rejection | 0.2 |
| Consequence answer names a concrete, specific harm | 0.2 |
| License answer correctly applies the reuse rules | 0.15 |
| **Total** | **1.0** |

On the consequence answer: "it would break" earns 0.05. "A negative temperature would put the
heater into a state the hardware cannot reach, and nobody would see an error" earns the full 0.2.
Specificity is the skill being graded, not length.

## Revision Log

The LxD cycle is not complete without a documented change driven by real feedback. Fill in
Revision 2 after teaching — do not write it in advance.

### Revision 1 — from design review, before teaching

**Feedback received:** *[who, when]*

**Change made:** The first draft opened with the definition of system reliability and reached the
`Thermostat` example afterward. That order asks students to accept a principle before they have
felt the problem. Revised so the hook comes first, the trace table with the wrong row comes second,
and the formal definition appears only after the class has already seen a method confidently return
garbage.

**Why:** The empathy finding was that students test to confirm success. A definition does not
disturb that habit; watching a passing test produce a wrong answer does.

### Revision 2 — from teaching day

**Feedback received:** *[fill in — what confused people, what a peer suggested]*

**Change made:** *[fill in]*

**Why:** *[fill in]*

## Self-Scoring and Evidence

Per the sprint framework, use 0.09 of 0.10 as a typical maximum per item.

### Hard Skills

| Item | Points | Self-Score | Evidence |
|---|---|---|---|
| Fundamentals lessons | .1 × 5 | | Running example: data types (`double`, `String`), conditionals in `getStatus`, methods, class, object |
| Example lesson: input and output | .1 | | Worked example and trace table |
| Object definitions | .1 | | `Thermostat` class, constructor, accessor, mutator |
| Language comparisons | .1 | | Exception vs boolean return (Popcorn Hacks, Part 2); `NaN` comparison behavior |
| Interesting hacks / real-world connection | .1 | | Parts 3 and 4 — connected thermostat at scale |
| LxD process documentation | .1 | | Lesson Design section and Revision Log |
| **Total** | **1** | | |

### Teaching and Collaboration

| Item | Points | Self-Score | Evidence |
|---|---|---|---|
| Teaching effectiveness | .1 × 3 | | Teaching-day notes or recording |
| Advocacy and collaboration | .1 × 3 | | GitHub issue link; feedback given on a peer's lesson |
| LxD design thinking application | .1 × 2 | | POV statement and HMW question |
| Peer feedback and iteration | .1 × 2 | | Revision Log, Revision 2 |
| **Total** | **1** | | |

## Lesson Closeout

Complete after teaching, before the live review.

**One learner need identified:** Students who can write a working method still treat invalid input
as the user's mistake rather than the method's responsibility, because they test to confirm success
rather than to find failure.

**One effective activity:** *[fill in — which activity actually produced understanding?]*

**One lesson revision:** *[fill in from Revision Log, Revision 2]*

**One team practice to improve next sprint:** *[fill in — for example, reviewing each other's
mutators for missing guards before merging, since this is easiest to catch in review]*

## Sources

- Source notebook adapted for this lesson: `_projects/lessons/java/notebooks/` — AP CSA Topic 3.2,
  Impact of Program Design *(update this path to the notebook you adapted)*.
- House style reference: `_notebooks/CSA/ap_mcq_lessons/unit_03/2026-09-15-3.9thisKeyword.ipynb`.
- College Board, *AP Computer Science A Course and Exam Description*, Unit 3, Topic 3.2.
- Oracle, "Class Double" — `isNaN` and the comparison behavior of `NaN`.
  <https://docs.oracle.com/en/java/javase/17/docs/api/java.base/java/lang/Double.html>
- Oracle, "Class IllegalArgumentException."
  <https://docs.oracle.com/en/java/javase/17/docs/api/java.base/java/lang/IllegalArgumentException.html>
- Oracle, *The Java Tutorials* — "Unchecked Exceptions: The Controversy."
  <https://docs.oracle.com/javase/tutorial/essential/exceptions/runtime.html>
- Open Source Initiative, "The Open Source Definition." <https://opensource.org/osd>
- GitHub Docs, "Licensing a repository" — on repositories published without a license.
  <https://docs.github.com/en/repositories/managing-your-repositorys-settings-and-features/customizing-your-repository/licensing-a-repository>